# Microsoft Planetary Computer 3DEP LiDAR COPC coverage

Investigates how current the `3dep-lidar-copc` collection on Microsoft Planetary Computer is,
compared to the live USGS WESM index and hobu's `usgs-lidar-public` EPT bucket.

Dataset page: https://planetarycomputer.microsoft.com/dataset/3dep-lidar-copc
STAC API: https://planetarycomputer.microsoft.com/api/stac/v1/collections/3dep-lidar-copc

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import pystac_client
import planetary_computer
import requests

STAC_API_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
COLLECTION_ID = "3dep-lidar-copc"

# pystac_client has no request timeout by default, which can hang indefinitely on a
# stalled connection; set one explicitly (connect timeout, read timeout).
catalog = pystac_client.Client.open(
    STAC_API_URL, modifier=planetary_computer.sign_inplace, timeout=(10, 30)
)

## 1. Collection-level metadata

Start with what the collection itself declares about its temporal extent, before querying items directly —
collection-level summaries can be stale, so this is a starting hypothesis to verify, not a conclusion.

In [ ]:
collection = catalog.get_collection(COLLECTION_ID)
print("title:", collection.title)
print("temporal extent:", collection.extent.temporal.intervals)
print("spatial bbox:", collection.extent.spatial.bboxes)

## 2. Items are per-tile, not per-project

Each STAC item is a single COPC tile (a spatial chunk of one USGS workunit/project), not one item per
project the way hobu's `resources.geojson` or WESM's `workunit` rows are. The property
`3dep:usgs_id` is the join key back to a WESM `workunit`/`project` name. Pull one sample item to confirm
structure and see the asset URL pattern.

In [ ]:
search = catalog.search(collections=[COLLECTION_ID], limit=1)
sample_item = next(search.items())
print("item id:", sample_item.id)
print("3dep:usgs_id:", sample_item.properties.get("3dep:usgs_id"))
print("start_datetime:", sample_item.properties.get("start_datetime"))
print("end_datetime:", sample_item.properties.get("end_datetime"))
print("data asset href:", sample_item.assets["data"].href)

## 3. Confirm the recency cutoff directly against the search API

Rather than trusting the collection's declared temporal extent, query for any items with a datetime
in 2023 or later. If none are returned, the collection genuinely has no recent coverage (not just an
outdated summary field).

In [ ]:
recent_search = catalog.search(
    collections=[COLLECTION_ID],
    datetime="2023-01-01/..",
    limit=1,
)
recent_items = list(recent_search.items())
print(f"items with datetime >= 2023-01-01: {len(recent_items)}")

## 4. Enumerate distinct projects (`3dep:usgs_id`) in the collection

There's no collection-level list of projects and no aggregation endpoint, so the only way to enumerate
projects is to page through tile-level items. **This collection is huge at the tile level** (a single
project can have thousands of tiles) but items are returned grouped by project, so a bounded page walk
still sees every tile of every project it reaches — it just may not reach every project in the whole
collection. Given ~1.2s per 1000-tile page and thousands of tiles per project, a full exhaustive walk is
impractically slow for a notebook cell, so this walks with a **time budget** instead of exhaustively, and
is explicit that results below the full collection size are a (still valid, contiguous) partial sample,
not full coverage. Results are cached to parquet so re-running the notebook is instant.

Uses raw `requests` against the search POST endpoint (not `pystac_client`) because requesting only a
subset of `fields` breaks pystac-client's strict item validation (STAC responses need `id`/`geometry`
etc. to construct a full `Item`, which the fields filter deliberately omits to save bandwidth).

In [ ]:
import time
from pathlib import Path

CACHE_PATH = Path("pc_3dep_projects_cache.parquet")
SEARCH_URL = f"{STAC_API_URL}/search"
TIME_BUDGET_SECONDS = 180  # walk the collection for up to this long, then stop


def enumerate_pc_projects(time_budget_seconds=TIME_BUDGET_SECONDS, page_limit=1000):
    """Walk the STAC item search (raw requests, paginated) and aggregate one row per
    distinct 3dep:usgs_id project seen within the time budget. Since items are grouped
    by project, every project this reaches is fully counted (no split tile counts)."""
    body = {
        "collections": [COLLECTION_ID],
        "limit": page_limit,
        "fields": {
            "include": [
                "properties.3dep:usgs_id",
                "properties.start_datetime",
                "properties.end_datetime",
            ]
        },
    }

    projects = {}
    t0 = time.time()
    n_pages = 0
    n_tiles = 0
    while True:
        resp = requests.post(SEARCH_URL, json=body, timeout=30)
        resp.raise_for_status()
        page = resp.json()
        for feat in page["features"]:
            props = feat["properties"]
            usgs_id = props.get("3dep:usgs_id")
            if usgs_id is None:
                continue
            entry = projects.setdefault(
                usgs_id, {"tile_count": 0, "start_datetime": None, "end_datetime": None}
            )
            entry["tile_count"] += 1
            start, end = props.get("start_datetime"), props.get("end_datetime")
            if start and (entry["start_datetime"] is None or start < entry["start_datetime"]):
                entry["start_datetime"] = start
            if end and (entry["end_datetime"] is None or end > entry["end_datetime"]):
                entry["end_datetime"] = end
        n_pages += 1
        n_tiles += len(page["features"])

        next_link = next((l for l in page["links"] if l["rel"] == "next"), None)
        elapsed = time.time() - t0
        if not next_link or elapsed >= time_budget_seconds:
            complete = next_link is None
            break
        body = next_link["body"]
        if n_pages % 20 == 0:
            print(f"...{elapsed:.0f}s: {n_pages} pages, {n_tiles} tiles, {len(projects)} distinct projects")

    df = pd.DataFrame.from_dict(projects, orient="index").reset_index(names="usgs_id")
    return df, complete, n_tiles


if CACHE_PATH.exists():
    pc_projects = pd.read_parquet(CACHE_PATH)
    print(f"loaded {len(pc_projects)} projects from cache: {CACHE_PATH}")
else:
    pc_projects, walk_complete, n_tiles_seen = enumerate_pc_projects()
    pc_projects.to_parquet(CACHE_PATH)
    status = "reached the end of the collection" if walk_complete else "stopped at the time budget (PARTIAL sample)"
    print(f"walked {n_tiles_seen} tiles, found {len(pc_projects)} distinct projects, {status}")
    print(f"cached to {CACHE_PATH} — delete this file to re-run the walk (e.g. with a larger time budget)")

pc_projects["end_datetime"] = pd.to_datetime(pc_projects["end_datetime"])
pc_projects["start_datetime"] = pd.to_datetime(pc_projects["start_datetime"])
pc_projects["collection_year"] = pc_projects["end_datetime"].dt.year
pc_projects.sort_values("end_datetime", ascending=False).head(10)

## 5. Year distribution of Planetary Computer projects

In [ ]:
year_counts = pc_projects["collection_year"].value_counts().sort_index()
ax = year_counts.plot(kind="bar", figsize=(10, 4), title="Planetary Computer 3DEP COPC projects by collection year")
ax.set_xlabel("collection year")
ax.set_ylabel("# projects")
plt.tight_layout()
plt.show()

print("most recent collection_year in Planetary Computer:", year_counts.index.max())
print("projects with collection_year >= 2023:", int(year_counts[year_counts.index >= 2023].sum()) if (year_counts.index >= 2023).any() else 0)

## 6. Cross-reference against the live WESM index

Pull the current WESM workunit list (same source used in `fetch_3dep_metadata.py`) and check, for each
recent year, what fraction of WESM workunits have a matching project in Planetary Computer — matched by
normalized name against `3dep:usgs_id` (which typically looks like `<workunit>_1` or similar, so match on
prefix/contains rather than requiring an exact string).

**Caveat if the walk above was a partial sample** (didn't reach the end of the collection): a
`False` here means "not found in the pages walked so far," not "confirmed absent from Planetary
Computer" — the true PC coverage could be somewhat higher. This is still useful for spot-checking
recent years, since large modern deliveries tend to be well-represented once the walk reaches their
region alphabetically/spatially, but treat the percentages as a lower bound, not exact.

In [ ]:
import io

# pd.read_csv on a bare URL has no timeout and can hang indefinitely if the connection
# stalls (rockyweb.usgs.gov has done this before — see fetch_3dep_metadata.py); fetch
# with requests (explicit timeout) first, then hand the bytes to read_csv.
wesm_resp = requests.get(
    "https://rockyweb.usgs.gov/vdelivery/Datasets/Staged/Elevation/metadata/WESM.csv",
    timeout=60,
)
wesm_resp.raise_for_status()
wesm = pd.read_csv(io.StringIO(wesm_resp.text), parse_dates=["collect_start", "collect_end"])


def wesm_year(row):
    if pd.notna(row["collect_start"]):
        return row["collect_start"].year
    elif pd.notna(row["collect_end"]):
        return row["collect_end"].year
    return None


wesm["collection_year"] = wesm.apply(wesm_year, axis=1)

pc_usgs_ids = set(pc_projects["usgs_id"])


def has_pc_match(workunit):
    # 3dep:usgs_id values look like '<workunit>' or '<workunit>_<n>' in practice;
    # check exact match and a loose prefix match to be permissive about the join.
    if workunit in pc_usgs_ids:
        return True
    return any(uid.startswith(workunit) or workunit.startswith(uid) for uid in pc_usgs_ids)


recent_wesm = wesm[wesm["collection_year"] >= 2020].copy()
recent_wesm["in_planetary_computer"] = recent_wesm["workunit"].apply(has_pc_match)

coverage_by_year = (
    recent_wesm.groupby("collection_year")["in_planetary_computer"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "n_in_pc", "count": "n_wesm_total"})
)
coverage_by_year["pct_covered"] = (100 * coverage_by_year["n_in_pc"] / coverage_by_year["n_wesm_total"]).round(1)
coverage_by_year

## 7. Compare all three sources side by side

For each recent WESM collection year: how many workunits, and what fraction are available via
hobu's EPT bucket (`usgs_3dep_resources.geojson`, built by `fetch_3dep_metadata.py`) vs. Planetary
Computer's COPC collection.

In [ ]:
hobu_resources = gpd.read_file(Path("usgs_3dep_resources.geojson"))
hobu_names = set(hobu_resources["name"])

recent_wesm["in_hobu_ept"] = recent_wesm["workunit"].isin(hobu_names)

comparison = (
    recent_wesm.groupby("collection_year")[["in_hobu_ept", "in_planetary_computer"]]
    .agg(["sum"])
)
comparison.columns = ["n_in_hobu_ept", "n_in_planetary_computer"]
comparison["n_wesm_total"] = recent_wesm.groupby("collection_year").size()
comparison["pct_hobu"] = (100 * comparison["n_in_hobu_ept"] / comparison["n_wesm_total"]).round(1)
comparison["pct_planetary_computer"] = (100 * comparison["n_in_planetary_computer"] / comparison["n_wesm_total"]).round(1)
comparison

In [ ]:
ax = comparison[["pct_hobu", "pct_planetary_computer"]].plot(
    kind="bar", figsize=(10, 5), title="% of WESM workunits available as streamable point cloud, by source"
)
ax.set_ylabel("% of WESM workunits covered")
ax.set_xlabel("collection year")
ax.legend(["hobu EPT (usgs-lidar-public)", "Planetary Computer (3dep-lidar-copc)"])
plt.tight_layout()
plt.show()

## Conclusion

Fill in after running: does Planetary Computer's `3dep-lidar-copc` collection extend past its
declared 2022-01-01 temporal extent cutoff, and how does its recent-year coverage compare to hobu's
EPT bucket? If both lag WESM significantly for 2023+ workunits, the practical implication is the same
as for hobu: recent collections are only accessible as raw staged LAZ (via `lpc_link` from the WESM
ArcGIS index), not yet as a cloud-optimized/streamable point cloud, regardless of provider.